<a href="https://colab.research.google.com/github/chavezaltamirano-ui/Growth-Models-in-Comparative/blob/main/4Growth_Models_in_Comparative_Perspective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

print("data_raw:", os.listdir("data_raw"))
print("data_clean:", os.listdir("data_clean"))

data_raw: []
data_clean: []


In [12]:
from google.colab import files
uploaded = files.upload()

Saving generar_cuadros_apa.py to generar_cuadros_apa.py


In [14]:
import os
import shutil

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

for fname in uploaded.keys():
    # Panel combinado lo dejamos en data_clean
    if fname == "generar_cuadros_apa.py":
        dst = os.path.join("data_clean", fname)
    else:
        # Bloques fuente en data_raw
        dst = os.path.join("data_raw", fname)

    shutil.move(fname, dst)
    print("Movido a", dst)

print("\nContenido de data_raw:", os.listdir("data_raw"))
print("Contenido de data_clean:", os.listdir("data_clean"))

FileNotFoundError: [Errno 2] No such file or directory: 'generar_cuadros_apa.py'

In [15]:
!ls -la data_clean/china_us_super_panel_1990_2025.csv

-rw-r--r-- 1 root root 24443 Jul 30 22:15 data_clean/china_us_super_panel_1990_2025.csv


In [16]:
!find / -name "china_us_super_panel_1990_2025.csv" 2>/dev/null

/content/data_raw/china_us_super_panel_1990_2025.csv
/content/data_clean/china_us_super_panel_1990_2025.csv


In [17]:
!find / -name "generar_cuadros_apa.py" 2>/dev/null

/content/data_clean/generar_cuadros_apa.py


In [18]:
!pip -q install python-docx statsmodels arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 25.3 MB/s eta 0:00:00


In [19]:
print("Listo")

Listo


In [20]:
import docx
import statsmodels
import arch

print("Todas las librerías están instaladas correctamente")

Todas las librerías están instaladas correctamente


In [21]:
!python /content/generar_cuadros_apa.py

python3: can't open file '/content/generar_cuadros_apa.py': [Errno 2] No such file or directory


In [22]:
from google.colab import files
uploaded = files.upload()

Saving generar_cuadros_apa.py to generar_cuadros_apa.py


In [23]:
import os

ruta = "/content/generar_cuadros_apa.py"
print(os.path.exists(ruta))

True


In [24]:
!python /content/generar_cuadros_apa.py

Panel cargado: 68 observaciones x 36 variables (1990-2023)
Cuadro 1 listo: cobertura y disponibilidad
Cuadro 2 listo: estadistica descriptiva
Cuadro 3 listo: pruebas de raiz unitaria
Cuadros 4, 5 y 6 listos: correlaciones y VIF

Documento Word generado en: /content/outputs/cuadros_previos_apa.docx
Tablas auxiliares en CSV dentro de: /content/outputs/


In [25]:
from google.colab import files
files.download("/content/outputs/cuadros_previos_apa.docx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# -*- coding: utf-8 -*-
"""
REPARACION Y AUDITORIA DE SERIES DEL PANEL MAESTRO
Proyecto: modelos de crecimiento comparados China - Estados Unidos, 1990-2023

NO modifica el panel maestro. Solo lo lee y escribe archivos derivados.

Salidas en /content/data_clean:
    china_us_super_panel_reparado.csv   panel con las series reconstruidas
    bitacora_reparacion.csv            registro de decisiones (cuadro 5)
    cobertura_series_nuevas.csv         cobertura de lo reconstruido

Uso en Colab:
    !python /content/reparar_series.py
"""

import json, os, time, urllib.request
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- CONFIGURACION
PANEL_MAESTRO = "/content/data_clean/china_us_super_panel_1990_2025.csv"
OUTDIR        = "/content/data_clean"
ANIO_INI, ANIO_FIN = 1990, 2024
ISO3 = {"CN": "CHN", "US": "USA"}

INDICADORES = {
    "va_agr_pct":     "NV.AGR.TOTL.ZS",   # VA agricola, % del PIB
    "va_ind_pct":     "NV.IND.TOTL.ZS",   # VA industrial, % del PIB
    "va_srv_pct":     "NV.SRV.TOTL.ZS",   # VA servicios, % del PIB
    "emp_agr_pct":    "SL.AGR.EMPL.ZS",   # empleo agricola, % del total
    "emp_ind_pct":    "SL.IND.EMPL.ZS",   # empleo industrial, % del total
    "emp_srv_pct":    "SL.SRV.EMPL.ZS",   # empleo servicios, % del total
    "fuerza_laboral": "SL.TLF.TOTL.IN",
    "desempleo_pct":  "SL.UEM.TOTL.ZS",
    "pib_real_usd":   "NY.GDP.MKTP.KD",   # PIB, USD constantes 2015
}

URL_WB = ("https://api.worldbank.org/v2/country/{iso}/indicator/{ind}"
          "?date={ini}:{fin}&format=json&per_page=500")
REINTENTOS, ESPERA = 3, 3


# ------------------------------------------------------------------- DESCARGA
def descargar_wb(iso, indicador):
    url = URL_WB.format(iso=iso, ind=indicador, ini=ANIO_INI, fin=ANIO_FIN)
    for intento in range(1, REINTENTOS + 1):
        try:
            with urllib.request.urlopen(url, timeout=60) as resp:
                datos = json.loads(resp.read().decode("utf-8"))
            if not isinstance(datos, list) or len(datos) < 2 or datos[1] is None:
                return pd.DataFrame(columns=["year", "value"])
            df = pd.DataFrame([(int(x["date"]), x["value"]) for x in datos[1]],
                              columns=["year", "value"])
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            return df.sort_values("year").reset_index(drop=True)
        except Exception as e:
            if intento == REINTENTOS:
                print("    fallo definitivo {} / {}: {}".format(iso, indicador, e))
                return pd.DataFrame(columns=["year", "value"])
            time.sleep(ESPERA)
    return pd.DataFrame(columns=["year", "value"])


def descargar_componentes():
    registros = []
    for cod, iso in ISO3.items():
        print("  Componentes de {} ({})".format(cod, iso))
        for nombre, indicador in INDICADORES.items():
            df = descargar_wb(iso, indicador)
            n = int(df["value"].notna().sum()) if len(df) else 0
            print("    {:16s} {:18s} n = {}".format(nombre, indicador, n))
            if n == 0:
                continue
            df = df.dropna(subset=["value"]).copy()
            df["country"], df["serie"] = cod, nombre
            registros.append(df[["country", "year", "serie", "value"]])
    if not registros:
        raise RuntimeError("No se descargo ningun componente. Revisa la conexion.")
    largo = pd.concat(registros, ignore_index=True)
    ancho = largo.pivot_table(index=["country", "year"], columns="serie",
                              values="value", aggfunc="first").reset_index()
    ancho.columns.name = None
    return ancho


# ------------------------------------------- RECONSTRUCCION DE PRODUCTIVIDAD
def reconstruir_productividad(comp):
    """VA por ocupado sectorial, USD constantes 2015, metodo identico CN / US."""
    df = comp.copy()
    for r in ["fuerza_laboral", "desempleo_pct", "pib_real_usd"]:
        if r not in df.columns:
            df[r] = np.nan

    df["ocupados_totales"] = df["fuerza_laboral"] * (1 - df["desempleo_pct"] / 100.0)

    for s in ["agr", "ind", "srv"]:
        cva, cemp = "va_{}_pct".format(s), "emp_{}_pct".format(s)
        for c in (cva, cemp):
            if c not in df.columns:
                df[c] = np.nan
        va_s  = (df[cva] / 100.0) * df["pib_real_usd"]
        emp_s = (df[cemp] / 100.0) * df["ocupados_totales"]
        with np.errstate(divide="ignore", invalid="ignore"):
            df["{}_va_per_worker_rec".format(s)] = np.where(
                (emp_s > 0) & np.isfinite(emp_s), va_s / emp_s, np.nan)

    # brechas relativas de productividad: indicadores de cambio estructural
    df["ratio_ind_agr_rec"] = df["ind_va_per_worker_rec"] / df["agr_va_per_worker_rec"]
    df["ratio_srv_ind_rec"] = df["srv_va_per_worker_rec"] / df["ind_va_per_worker_rec"]

    nuevas = ["agr_va_per_worker_rec", "ind_va_per_worker_rec",
              "srv_va_per_worker_rec", "ratio_ind_agr_rec",
              "ratio_srv_ind_rec", "ocupados_totales"]
    return df[["country", "year"] + nuevas], nuevas


# -------------------------------------------------------------------- AUDITORIA
def cobertura(df, variables, etiqueta=""):
    filas, anios = [], int(df["year"].max() - df["year"].min() + 1)
    for v in variables:
        if v not in df.columns:
            continue
        for pais in ["CN", "US"]:
            sub = df.loc[df["country"] == pais, ["year", v]].dropna()
            if len(sub) == 0:
                filas.append([etiqueta, v, pais, "-", "-", 0, 100.0])
            else:
                filas.append([etiqueta, v, pais, int(sub["year"].min()),
                              int(sub["year"].max()), len(sub),
                              round(100.0 * (1 - len(sub) / float(anios)), 1)])
    return pd.DataFrame(filas, columns=["Conjunto", "Variable", "Pais",
                                        "Anio_inicio", "Anio_fin", "n",
                                        "Pct_faltantes"])


# ------------------------------------------- BITACORA DE TRATAMIENTO (CUADRO 5)
BITACORA_FILAS = [
 ["inv_gfcf_const_usd", "China",
  "Serie con una sola observacion (2015).",
  "Variable excluida; se conserva inv_gfcf_gdp.",
  "El WDI no publica NE.GDI.FTOT.KD para China fuera del ano base 2015 ni "
  "NE.GDI.FTOT.KN en moneda local constante (verificado en la API). La FBCF "
  "como porcentaje del PIB cubre 1990-2023 sin interrupciones y es la medida "
  "usada en la literatura de regimenes de crecimiento."],

 ["agr/ind/srv_va_per_worker", "Estados Unidos",
  "Series con una sola observacion (2015).",
  "Series reconstruidas con metodo identico para ambos paises (sufijo _rec).",
  "NV.*.EMPL.KD solo reporta el ano base para Estados Unidos. Se reconstruye el "
  "VA por ocupado a partir de la participacion sectorial en el valor agregado, "
  "el PIB real en USD constantes de 2015, la participacion sectorial en el "
  "empleo y los ocupados estimados. El mismo procedimiento en los dos paises "
  "asegura comparabilidad. Cobertura efectiva: 1997-2021."],

 ["gov_debt_gdp", "Ambos",
  "Solo 7 observaciones (2017-2023).",
  "Variable excluida del analisis econometrico.",
  "La serie del WEO incorporada al panel no cubre el periodo. La deuda publica "
  "se mide con pub_debt_gdp de la Global Debt Database del FMI, conforme a la "
  "jerarquia de fuentes declarada."],

 ["hh_debt_gdp / corp_debt_gdp / priv_debt_gdp", "China",
  "Cero observaciones en el periodo.",
  "Bloque financiero de China operacionalizado con bis_tot_credit_gdp y "
  "bis_pvt_credit_gdp; desagregacion sectorial reservada a Estados Unidos.",
  "La Global Debt Database del FMI cubre la deuda privada sectorial de China "
  "continental solo desde 2006 y el BIS publica credito a hogares y a "
  "sociedades no financieras de China desde el primer trimestre de 2006. "
  "Limitacion de fuentes, no decision analitica."],

 ["credit_finsec_gdp", "China",
  "Cero observaciones.",
  "Variable excluida; el bloque financiero se mide con series del BIS.",
  "El indicador de activos del sistema financiero del WDI no esta disponible "
  "para China en el periodo de estudio."],

 ["bis_hh_credit_gdp / bis_nfc_credit_gdp", "China",
  "17 observaciones (desde 2005).",
  "Uso restringido a estadistica descriptiva y subperiodos; excluidas del ARDL.",
  "Con 17 observaciones anuales no es posible especificar un modelo "
  "autorregresivo con rezagos distribuidos y controles."],

 ["Series del BIS (todas)", "Ambos",
  "Cobertura que termina en 2021.",
  "Las especificaciones con credito del BIS se estiman sobre 1990-2021, y sobre "
  "1994-2021 para el credito total de China.",
  "La muestra efectiva se reporta en cada modelo para evitar comparaciones "
  "entre periodos distintos."],

 ["labour_prod; pub_debt_gdp (CN); hh_debt_gdp (US)", "Ambos",
  "Clasificadas I(2) con especificacion de solo constante.",
  "Reestimacion con constante y tendencia, logaritmos y prueba de quiebre "
  "estructural endogeno.",
  "En series con tendencia determinista la especificacion sin tendencia esta "
  "mal identificada. La discrepancia entre ADF y Phillips-Perron en la deuda "
  "publica de China sugiere quiebre estructural, no integracion de orden dos."],
]


# ------------------------------------------------------------------- EJECUCION
def main(panel_maestro=PANEL_MAESTRO, outdir=OUTDIR):
    os.makedirs(outdir, exist_ok=True)
    f_panel = os.path.join(outdir, "china_us_super_panel_reparado.csv")
    f_bit   = os.path.join(outdir, "bitacora_reparacion.csv")
    f_cob   = os.path.join(outdir, "cobertura_series_nuevas.csv")

    if not os.path.exists(panel_maestro):
        raise FileNotFoundError("No se encontro el panel: " + panel_maestro)

    print("1. Leyendo el panel maestro (solo lectura)")
    master = pd.read_csv(panel_maestro)
    master.columns = [str(c).strip() for c in master.columns]
    master["country"] = master["country"].astype(str).str.strip().str.upper()
    master["year"] = pd.to_numeric(master["year"], errors="coerce").astype(int)
    print("   {} observaciones x {} variables".format(*master.shape))

    print("\n2. Descargando componentes del Banco Mundial")
    comp = descargar_componentes()

    print("\n3. Reconstruyendo la productividad sectorial")
    rec, nuevas = reconstruir_productividad(comp)

    print("\n4. Integrando en un panel derivado")
    panel = master.merge(rec, on=["country", "year"], how="left")
    excluir = [c for c in ["gov_debt_gdp", "inv_gfcf_const_usd",
                           "credit_finsec_gdp"] if c in panel.columns]
    panel = panel.rename(columns={c: c + "_excluida" for c in excluir})
    panel.to_csv(f_panel, index=False)
    print("   {}  ->  {} obs x {} vars".format(f_panel, *panel.shape))

    print("\n5. Cobertura de las series reconstruidas")
    cob = cobertura(panel, nuevas, "Series reconstruidas")
    cob.to_csv(f_cob, index=False)
    print(cob.to_string(index=False))

    print("\n6. Bitacora de tratamiento")
    bit = pd.DataFrame(BITACORA_FILAS, columns=["Variable", "Pais", "Incidencia",
                                                "Decision", "Justificacion"])
    bit.to_csv(f_bit, index=False)
    print("   {} ({} registros)".format(f_bit, len(bit)))

    print("\nListo. El panel maestro NO fue modificado.")
    return f_panel


if __name__ == "__main__":
    main()

1. Leyendo el panel maestro (solo lectura)
   68 observaciones x 36 variables

2. Descargando componentes del Banco Mundial
  Componentes de CN (CHN)
    va_agr_pct       NV.AGR.TOTL.ZS     n = 35
    va_ind_pct       NV.IND.TOTL.ZS     n = 35
    va_srv_pct       NV.SRV.TOTL.ZS     n = 35
    emp_agr_pct      SL.AGR.EMPL.ZS     n = 34
    emp_ind_pct      SL.IND.EMPL.ZS     n = 34
    emp_srv_pct      SL.SRV.EMPL.ZS     n = 34
    fuerza_laboral   SL.TLF.TOTL.IN     n = 35
    desempleo_pct    SL.UEM.TOTL.ZS     n = 34
    pib_real_usd     NY.GDP.MKTP.KD     n = 35
  Componentes de US (USA)
    va_agr_pct       NV.AGR.TOTL.ZS     n = 25
    va_ind_pct       NV.IND.TOTL.ZS     n = 25
    va_srv_pct       NV.SRV.TOTL.ZS     n = 25
    emp_agr_pct      SL.AGR.EMPL.ZS     n = 34
    emp_ind_pct      SL.IND.EMPL.ZS     n = 34
    emp_srv_pct      SL.SRV.EMPL.ZS     n = 34
    fuerza_laboral   SL.TLF.TOTL.IN     n = 35
    desempleo_pct    SL.UEM.TOTL.ZS     n = 34
    pib_real_usd     NY.G